# La loi d'Okun : chômage et croissance · *Okun's law: unemployment and growth*

Notebook compagnon du chapitre **15. Croissance potentielle et output gap : la carte que personne ne peut mesurer** — [lire l'article](https://nmlab.io/ressources/croissance-potentielle-et-output-gap).
Companion notebook to chapter **15. Potential Growth and the Output Gap: The Map No One Can Measure** — [read the article](https://nmlab.io/en/ressources/potential-growth-and-output-gap).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


import pandas as pd
from pandas import DataFrame

def load_okun() -> DataFrame:
    """Par année : variation du taux de chômage (UNRATE) et croissance du PIB réel (GDPC1),
    en moyennes annuelles depuis FRED. / Yearly change in unemployment and real GDP growth."""
    growth = nm.load_fred("GDPC1").resample("YS").mean().pct_change() * 100
    du = nm.load_fred("UNRATE").resample("YS").mean().diff()
    return pd.DataFrame({"du": du, "growth": growth}).dropna()

okun = load_okun()


import numpy as np
from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="La loi d'Okun : chômage et croissance",
        sub="Chaque point = une année, États-Unis, 1949-2025",
        xlab="variation du taux de chômage sur l'année, points",
        ylab="croissance du PIB réel, %",
        slope="pente ≈ −1,5", l1="1 point de chômage", l2="≈ 1,5 point de PIB",
        note="Quand le chômage monte, la croissance faiblit — d'environ 1,5 point de PIB par point de chômage. Une\n"
             "régularité robuste, mais pas une « loi » : le coefficient varie. Source : BEA et BLS via FRED (GDPC1, UNRATE)."),
    "en": dict(
        title="Okun's law: unemployment and growth",
        sub="Each dot = one year, United States, 1949-2025",
        xlab="change in the unemployment rate over the year, points",
        ylab="real GDP growth, %",
        slope="slope ≈ −1.5", l1="1 point of unemployment", l2="≈ 1.5 point of GDP",
        note="When unemployment rises, growth weakens — by about 1.5 points of GDP per point of unemployment. A robust\n"
             "regularity, but not a « law »: the coefficient varies. Source: BEA and BLS via FRED (GDPC1, UNRATE)."),
}

def build_figure(okun: "DataFrame", lang: str) -> Figure:
    """Nuage annuel Δchômage vs croissance du PIB, avec droite de régression."""
    text = LABELS[lang]
    x, y = okun["du"].to_numpy(), okun["growth"].to_numpy()
    fig = nm.figure(height_px=1045)
    ax = nm.axes(fig, left=0.072, bottom=0.185)
    ax.axhline(0, color=nm.COLORS["muted"], linewidth=1.6, alpha=0.78)
    ax.axvline(0, color=nm.COLORS["muted"], linewidth=1.6, alpha=0.78)
    ax.scatter(x, y, s=70, color=nm.COLORS["blue"], alpha=0.8, linewidths=0, zorder=3)
    slope, intercept = np.polyfit(x, y, 1)
    line_x = np.array([-1.7, 3.2])
    ax.plot(line_x, intercept + slope * line_x, color=nm.COLORS["amber"], linewidth=4.0, zorder=4)
    ax.set_xlim(-1.7, 3.2)
    ax.set_xticks(range(-1, 4))
    ax.set_ylim(-4.3, 9.2)
    ax.set_yticks(range(-4, 9, 2))
    ax.set_xlabel(text["xlab"])
    ax.set_ylabel(text["ylab"])
    ax.text(1.55, 6.15, text["slope"], ha="center", fontsize=27, fontweight="bold", color=nm.COLORS["amber"])
    ax.text(1.55, 5.2, text["l1"], ha="center", fontsize=22, color=nm.COLORS["muted"])
    ax.text(1.55, 4.45, text["l2"], ha="center", fontsize=22, color=nm.COLORS["muted"])
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(okun, LANG)